<a href="https://colab.research.google.com/github/LeJ-04/web-datamining-semantics/blob/partieC-jeffrey/Part%20C/notebooks/C_KGE_Reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. Reasoning & Knowledge Graph Embeddings (KGE)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 0 — Configuration Google Drive (toujours en 1er)
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os

# ← Ton dossier Drive mis à jour
WORK_DIR = "/content/drive/MyDrive/Part C/Project Web Datamining & Semantics"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"✅ Drive monté.")
print(f"   Répertoire actif : {os.getcwd()}")
print(f"   Fichiers présents : {os.listdir(WORK_DIR)}")

In [ ]:
# # ── Migration unique — copie tout vers le Drive ──────────
# # À exécuter UNE SEULE FOIS pour transférer les fichiers existants
# import shutil, os

# fichiers_a_migrer = [
#     "clean_football_kb.ttl",
#     "clean_football_kb_typed.xml",
#     "augmented_football_kb.ttl",
#     "augmented_football_kb.xml",
#     "kge_metrics_comparison.csv",
#     "kge_metrics_full.csv",
#     "kge_training_loss.png"
# ]
# dossiers_a_migrer = ["kge_transe", "kge_complex", "kge_dataset"]

# SRC = "/content"  # répertoire Colab actuel
# DST = "/content/drive/MyDrive/Project Web Datamining & Semantics"

# for f in fichiers_a_migrer:
#     src = os.path.join(SRC, f)
#     if os.path.exists(src):
#         shutil.copy2(src, DST)
#         print(f"  ✓ {f}")
#     else:
#         print(f"  ⚠️  {f} non trouvé")

# for d in dossiers_a_migrer:
#     src = os.path.join(SRC, d)
#     dst = os.path.join(DST, d)
#     if os.path.exists(src):
#         shutil.copytree(src, dst, dirs_exist_ok=True)
#         print(f"  ✓ {d}/")
#     else:
#         print(f"  ⚠️  {d}/ non trouvé")

# print("\n✅ Migration terminée — tous les fichiers sont sur ton Drive.")

## C.1 Preparation of KGE Datasets (Train / Valid / Test)

Loading, Filtering and Splitting the Knowledge Graph

Knowledge Graph Embedding (KGE) models (such as TransE or ComplEx) learn vector representations of entities and relations based purely on the graph's topology.

To prepare our data for these models, we need to perform four critical operations.

1. **Filtering:**  
 KGE models strictly require **relational triples** `(head, relation, tail)` where all elements are URIs (Unique Resource Identifiers). We must filter out **Literals** (e.g., strings, numbers) and **Blank Nodes**.

In [ ]:
!pip install rdflib

In [ ]:
import os
import random
from rdflib import Graph, URIRef

In [ ]:
graph_file = "clean_football_kb.ttl"
print(f"Chargement du graphe depuis {graph_file}...")
g = Graph()
g.parse(graph_file, format="turtle")
print(f"Graphe chargé ! Nombre total de triplets dans le graphe : {len(g)}")

**Filtrage (isinstance(..., URIRef)) :**   
Les modèles mathématiques de KGE (Knowledge Graph Embeddings) comme TransE apprennent la structure du graphe. Ils ne savent pas lire du texte brut ou des nombres. On s'assure donc de ne garder que des triplets contenant des Identifiants (URI), c'est-à-dire des relations purement structurelles (ex: Joueur -> jouePour -> Equipe), en excluant les attributs comme les noms ou les âges (qui sont des Literal).

In [ ]:
# 2. Extraction et filtrage des triplets
# Les modèles KGE classiques n'apprennent généralement que sur des relations entre Entités (URIs).
# On filtre donc pour exclure les valeurs littérales (noms en texte, nombres, etc.) et les BNodes.
triplets = []
for s, p, o in g:
    if isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef):
        # We store them as tuples for easier manipulation during OOV checking
        triplets.append((str(s), str(p), str(o)))

print(f"Total relational triples retained for KGE: {len(triplets)}")

2. **Shuffling & Splitting:** We randomly shuffle the triples and split them into three sets:
   - **Training set (~80%):** Used by the model to learn the entity and relation embeddings.
   - **Validation set (~10%):** Used to tune hyperparameters and prevent overfitting.
   - **Testing set (~10%):** Held out until the end to evaluate the model's true performance.

In [ ]:
# 3. Mélange aléatoire (Shuffle) pour garantir une distribution homogène
random.seed(42) # Fixer la seed pour la reproductibilité de l'expérience
random.shuffle(triplets)

**Le Split (80/10/10) :**   
C'est le standard en Machine Learning. Le Train sert à l'apprentissage, le Valid permet de vérifier en cours de route que le modèle ne fait pas de surapprentissage (overfitting), et le Test est utilisé à la toute fin pour évaluer sa performance réelle sur des données qu'il n'a jamais vues.

In [ ]:
# 4. Calcul des indices pour la répartition 80% / 10% / 10%
total_triplets = len(triplets)
train_idx = int(total_triplets * 0.8)
valid_idx = int(total_triplets * 0.9)

train_triplets = triplets[:train_idx]
valid_triplets = triplets[train_idx:valid_idx]
test_triplets = triplets[valid_idx:]

print(f"Répartition des données :")
print(f" - Train : {len(train_triplets)} triplets")
print(f" - Valid : {len(valid_triplets)} triplets")
print(f" - Test  : {len(test_triplets)} triplets")

3. **Preventing Out-Of-Vocabulary (OOV) Entities:**  
 A KGE model cannot evaluate an entity or relation it has never seen during training. After the initial split, we systematically check the Validation and Test sets. Any triple containing an entity or relation missing from the Training set is moved into the Training set. This guarantees 100% graph connectivity during evaluation.

In [ ]:
# --- STEP 3: OUT-OF-VOCABULARY (OOV) PREVENTION ---
def get_vocab(triplet_list):
    """Returns a set of all entities and all relations in a given list of triplets."""
    entities, relations = set(), set()
    for s, p, o in triplet_list:
        entities.add(s)
        entities.add(o)
        relations.add(p)
    return entities, relations

In [ ]:
train_entities, train_relations = get_vocab(train_triplets)

In [ ]:
def fix_oov(target_triplets, train_triplets, train_entities, train_relations):
    """Moves triplets with unseen entities/relations to the train set."""
    fixed_target = []
    moved_count = 0
    for s, p, o in target_triplets:
        if s not in train_entities or o not in train_entities or p not in train_relations:
            # Missing in train: move this triplet to train
            train_triplets.append((s, p, o))
            train_entities.add(s)
            train_entities.add(o)
            train_relations.add(p)
            moved_count += 1
        else:
            fixed_target.append((s, p, o))
    return fixed_target, moved_count

In [ ]:
# Fix Validation Set
valid_triplets, moved_val = fix_oov(valid_triplets, train_triplets, train_entities, train_relations)
# Fix Test Set
test_triplets, moved_test = fix_oov(test_triplets, train_triplets, train_entities, train_relations)

In [ ]:
print(f"\nOOV Prevention Fixes:")
print(f" - Moved {moved_val} triples from Valid to Train.")
print(f" - Moved {moved_test} triples from Test to Train.")

print(f"\nFinal Data split breakdown:")
print(f" - Train : {len(train_triplets)} triples")
print(f" - Valid : {len(valid_triplets)} triples")
print(f" - Test  : {len(test_triplets)} triples")

4. **Exporting:**  
 We save the splits into `.txt` files in a Tab-Separated Values (`\t`) format, which is the standard input format expected by KGE libraries like PyKEEN.

In [ ]:
# --- STEP 4: FORMAT AND EXPORT THE DATASETS ---
def format_tsv(triplet_list):
    return [f"{s}\t{p}\t{o}\n" for s, p, o in triplet_list]

**L'export en .txt :**  
 Les bibliothèques de Machine Learning pour les graphes (comme PyKEEN) détestent parser de lourds fichiers XML ou Turtle. Elles demandent des fichiers texte basiques séparés par des tabulations (\t).

In [ ]:
# 5. Création du dossier et sauvegarde des fichiers
dataset_dir = "kge_dataset"
os.makedirs(dataset_dir, exist_ok=True)

with open(os.path.join(dataset_dir, "train.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(train_triplets))

with open(os.path.join(dataset_dir, "valid.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(valid_triplets))

with open(os.path.join(dataset_dir, "test.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(test_triplets))

print(f"\n✅ Successfully generated train.txt, valid.txt, and test.txt in the '{dataset_dir}/' directory.")

## C.2 - Symbolic Reasoning with SWRL and Owlready2
=================================================  


Knowledge Graphs allow for explicitly defining rules to infer new, hidden
knowledge not directly stated in the dataset. This is called Symbolic Reasoning
and represents the "Neuro-Symbolic" aspect of modern AI.

We use the `owlready2` library to declare SWRL rules and rdflib SPARQL CONSTRUCT
to execute them (Pellet, the native SWRL reasoner, requires Java 25+ which may
not always be available).

Business Rules:
  1. If a Person plays for a Team, and that Team has a headCoach C,
     then the Person is coachedBy C.
     SWRL: Person(?p) ∧ Team(?t) ∧ Person(?c) ∧ playsFor(?p,?t) ∧ headCoach(?t,?c)
           → coachedBy(?p,?c)

  2. If a Person plays for a Team, and that Team is locatedIn a Country,
     then the Person competesIn that Country.
     SWRL: Person(?p) ∧ Team(?t) ∧ Country(?c) ∧ playsFor(?p,?t) ∧ locatedIn(?t,?c)
           → competesIn(?p,?c)

These inferred properties enrich the Knowledge Graph and can later be used
by the LLM in the RAG pipeline (e.g. "Which players compete in England?").

In [ ]:
!pip install owlready2

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import rdflib
from rdflib import RDF, OWL, Namespace
from rdflib.namespace import RDFS
from owlready2 import World, Thing, ObjectProperty, Imp
import os

In [ ]:
# =====================================================================
# --- 1. PRÉPARATION DU GRAPHE (RDFLIB) ---
# =====================================================================
print("=" * 60)
print("C.2 — Symbolic Reasoning with SWRL and Owlready2")
print("=" * 60)
print("\n1. Chargement et préparation du graphe Turtle...")

g = rdflib.Graph()
g.parse("clean_football_kb.ttl", format="turtle")

# IRIs exactes du Knowledge Graph football
ONTO_IRI = "http://www.example.org/football"
EX       = Namespace("http://www.example.org/football/")       # entités
EX_PROP  = Namespace("http://www.example.org/football/prop/")  # propriétés

# Déclaration OWL de l'ontologie
g.add((URIRef(ONTO_IRI), RDF.type, OWL.Ontology))

# Déclaration explicite des classes OWL
for cls in ["Person", "Team", "Country", "FootballPosition"]:
    g.add((EX[cls], RDF.type, OWL.Class))

# Déclaration des propriétés (existantes + inférées)
for p in ["playsFor", "headCoach", "locatedIn",
          "nationality", "playsPosition", "coachedBy", "competesIn"]:
    g.add((EX_PROP[p], RDF.type, OWL.ObjectProperty))

# Domaine / portée des propriétés inférées
g.add((EX_PROP["coachedBy"],  RDFS.domain, EX["Person"]))
g.add((EX_PROP["coachedBy"],  RDFS.range,  EX["Person"]))
g.add((EX_PROP["competesIn"], RDFS.domain, EX["Person"]))
g.add((EX_PROP["competesIn"], RDFS.range,  EX["Country"]))

# Typage de toutes les instances comme OWL.NamedIndividual (requis par owlready2)
IGNORE = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty,
          OWL.AnnotationProperty, OWL.Ontology, OWL.NamedIndividual}
for s, p, o in list(g.triples((None, RDF.type, None))):
    if o not in IGNORE and isinstance(s, URIRef):
        g.add((s, RDF.type, OWL.NamedIndividual))

count_ni = len(list(g.triples((None, RDF.type, OWL.NamedIndividual))))
print(f"   -> {count_ni} entités taguées NamedIndividual")

# Contrôle des propriétés clés
for prop in ["playsFor", "locatedIn", "headCoach"]:
    triples = list(g.triples((None, EX_PROP[prop], None)))
    ex = triples[0] if triples else None
    print(f"   -> '{prop}' : {len(triples)} triplets" +
          (f"  (ex: {ex[0].split('/')[-1]} → {ex[2].split('/')[-1]})" if ex else " ⚠️ AUCUN"))

# Sérialisation en XML/OWL (format attendu par owlready2)
xml_file = os.path.abspath("clean_football_kb_typed.xml")
g.serialize(xml_file, format="xml")
print(f"   -> XML intermédiaire : {xml_file}")

In [ ]:
# =====================================================================
# --- 2. INITIALISATION DU SCHÉMA OWLREADY2 ---
# =====================================================================
print("\n2. Initialisation du schéma Owlready2...")

my_world = World()
onto     = my_world.get_ontology(f"file://{xml_file}").load()

# Résolution des classes (les noms courts sont utilisables dans les règles SWRL)
Person  = onto.search_one(iri=str(EX["Person"]))
Team    = onto.search_one(iri=str(EX["Team"]))
Country = onto.search_one(iri=str(EX["Country"]))

if not all([Person, Team, Country]):
    print("   ⚠️  Classes non trouvées. Diagnostic des 5 premières IRIs :")
    for i, e in enumerate(list(onto.individuals())[:5]):
        print(f"     {e.iri}")
    raise RuntimeError("Vérifiez que les namespaces EX et EX_PROP sont corrects.")

print(f"   -> Person={Person.name} | Team={Team.name} | Country={Country.name}")

# Namespace des propriétés (différent du base_iri de l'ontologie)
prop_ns = my_world.get_namespace("http://www.example.org/football/prop/")

In [ ]:
# =====================================================================
# --- 3. DÉCLARATION DES RÈGLES SWRL ---
# =====================================================================
print("\n3. Déclaration des règles SWRL (owlready2.Imp)...")

with onto:
    # Règle 1 — Inférer le coach d'un joueur
    rule1 = Imp()
    rule1.set_as_rule(
        "Person(?p), Team(?t), Person(?c), "
        "playsFor(?p, ?t), headCoach(?t, ?c) -> coachedBy(?p, ?c)",
        namespaces=[onto, prop_ns]
    )
    # Règle 2 — Inférer le pays de compétition d'un joueur
    rule2 = Imp()
    rule2.set_as_rule(
        "Person(?p), Team(?t), Country(?c), "
        "playsFor(?p, ?t), locatedIn(?t, ?c) -> competesIn(?p, ?c)",
        namespaces=[onto, prop_ns]
    )

print("   -> Règle 1 : Person(?p) ∧ Team(?t) ∧ Person(?c)  ∧ playsFor(?p,?t) ∧ headCoach(?t,?c) → coachedBy(?p,?c)")
print("   -> Règle 2 : Person(?p) ∧ Team(?t) ∧ Country(?c) ∧ playsFor(?p,?t) ∧ locatedIn(?t,?c) → competesIn(?p,?c)")
print("   ✓ 2 règles SWRL enregistrées dans l'ontologie.")

In [ ]:
# =====================================================================
# --- 4. APPLICATION DES RÈGLES (SPARQL CONSTRUCT) ---
# =====================================================================
# owlready2 délègue l'exécution des règles SWRL au raisonneur Pellet (Java).
# Pellet requiert Java 25+ (class file 69.0). Si cette version n'est pas
# disponible, on applique les mêmes règles via SPARQL CONSTRUCT sur rdflib,
# ce qui est sémantiquement équivalent — c'est d'ailleurs ce que Pellet
# effectue en interne sur les données RDF.
print("\n4. Application des règles via SPARQL CONSTRUCT...")

g.bind("ex",   EX)
g.bind("prop", EX_PROP)
g.bind("owl",  OWL)

SPARQL_RULE1 = """
PREFIX ex:   <http://www.example.org/football/>
PREFIX prop: <http://www.example.org/football/prop/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
CONSTRUCT { ?p prop:coachedBy ?c . }
WHERE {
    ?p  rdf:type       ex:Person .
    ?t  rdf:type       ex:Team   .
    ?c  rdf:type       ex:Person .
    ?p  prop:playsFor  ?t .
    ?t  prop:headCoach ?c .
}
"""

SPARQL_RULE2 = """
PREFIX ex:   <http://www.example.org/football/>
PREFIX prop: <http://www.example.org/football/prop/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
CONSTRUCT { ?p prop:competesIn ?c . }
WHERE {
    ?p  rdf:type        ex:Person  .
    ?t  rdf:type        ex:Team    .
    ?c  rdf:type        ex:Country .
    ?p  prop:playsFor   ?t .
    ?t  prop:locatedIn  ?c .
}
"""

# FIX : séparer query / add / count pour un décompte correct
new_triples_1 = list(g.query(SPARQL_RULE1))
for triple in new_triples_1:
    g.add(triple)
count_coached = len(new_triples_1)

new_triples_2 = list(g.query(SPARQL_RULE2))
for triple in new_triples_2:
    g.add(triple)
count_competes = len(new_triples_2)

print(f"   ✓ Règle 1 (coachedBy)  → {count_coached}  nouveaux triplets inférés")
print(f"   ✓ Règle 2 (competesIn) → {count_competes} nouveaux triplets inférés")

In [ ]:
# =====================================================================
# --- 5. VÉRIFICATION DES CONNAISSANCES INFÉRÉES ---
# =====================================================================
print("\n5. Vérification des nouvelles connaissances inférées...")

print("\n   ── Exemples : competesIn ──")
q_ex_competes = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?player ?country WHERE { ?player prop:competesIn ?country . } LIMIT 5
"""
for row in g.query(q_ex_competes):
    player  = str(row.player).split('/')[-1]
    country = str(row.country).split('/')[-1]
    print(f"     {player} compète en {country}")

print("\n   ── Exemples : coachedBy ──")
q_ex_coached = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?player ?coach WHERE { ?player prop:coachedBy ?coach . } LIMIT 5
"""
for row in g.query(q_ex_coached):
    player = str(row.player).split('/')[-1]
    coach  = str(row.coach).split('/')[-1]
    print(f"     {player} est coaché par {coach}")

print("\n   ── Joueurs par pays de compétition (top 5) ──")
q_by_country = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?country (COUNT(?player) AS ?nb)
WHERE { ?player prop:competesIn ?country . }
GROUP BY ?country ORDER BY DESC(?nb) LIMIT 5
"""
for row in g.query(q_by_country):
    country = str(row.country).split('/')[-1]
    print(f"     {country} : {int(row.nb)} joueurs")

In [ ]:
# =====================================================================
# --- 6. SAUVEGARDE DU GRAPHE AUGMENTÉ ---
# =====================================================================
print("\n6. Sauvegarde du graphe enrichi...")

augmented_ttl = "augmented_football_kb.ttl"
augmented_xml = "augmented_football_kb.xml"

g.serialize(augmented_ttl, format="turtle")   # format humain
onto.save(file=augmented_xml, format="rdfxml")  # format owlready2

total_triples = len(list(g.triples((None, None, None))))

print(f"""
{'='*60}
✅  C.2 TERMINÉ — Résumé du raisonnement symbolique
{'='*60}
  Graphe initial         : {total_triples - count_coached - count_competes} triplets
  Règles SWRL déclarées  : 2 (owlready2.Imp)
  Nouveaux coachedBy     : {count_coached}
  Nouveaux competesIn    : {count_competes}
  ─────────────────────────────────────────────
  Total triplets final   : {total_triples}
  Fichier Turtle         : {augmented_ttl}  ({os.path.getsize(augmented_ttl):,} octets)
  Fichier XML/OWL        : {augmented_xml}  ({os.path.getsize(augmented_xml):,} octets)
""")

## C.3 — Knowledge Graph Embeddings : Training TransE & ComplEx
# =====================================================================
PyKEEN est la librairie de référence pour les KGE en Python.
Elle gère nativement : chargement TSV, entraînement, évaluation, sauvegarde des embeddings et calcul des métriques.

Deux modèles entraînés :
- TransE   — modèle translationnel (h + r ≈ t dans l'espace vectoriel)
- ComplEx  — modèle bilinéaire dans l'espace complexe (meilleur sur les relations asymétriques comme playsFor, coachedBy)

In [ ]:
# Installation (à exécuter une seule fois) ─────────────────────────
!pip install pykeen torch --quiet

In [ ]:
import torch
import pykeen
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.models import TransE, ComplEx
import pandas as pd
import matplotlib.pyplot as plt
import os, json, time

# print(f"PyKEEN version : {pykeen.__version__}")
# print(f"PyTorch version : {torch.__version__}")
print(f"GPU disponible  : {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé  : {device}")

In [ ]:
# =====================================================================
# --- 1. CHARGEMENT DES SPLITS ---
# =====================================================================
print("\n1. Chargement des splits KGE...")

# Chemins vers les fichiers générés en C.1
TRAIN_PATH = "kge_dataset/train.txt"
VALID_PATH  = "kge_dataset/valid.txt"
TEST_PATH   = "kge_dataset/test.txt"

# TriplesFactory lit directement le format TSV (head \t relation \t tail)
# entity_to_id et relation_to_id sont partagés entre les 3 splits
# pour garantir la cohérence du vocabulaire (pas d'entités OOV)
tf_train = TriplesFactory.from_path(
    TRAIN_PATH,
    delimiter="\t"
)
tf_valid = TriplesFactory.from_path(
    VALID_PATH,
    delimiter="\t",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)
tf_test = TriplesFactory.from_path(
    TEST_PATH,
    delimiter="\t",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)

print(f"   Train  : {tf_train.num_triples:>6,} triplets")
print(f"   Valid  : {tf_valid.num_triples:>6,} triplets")
print(f"   Test   : {tf_test.num_triples:>6,} triplets")
print(f"   Entités    : {tf_train.num_entities:,}")
print(f"   Relations  : {tf_train.num_relations:,}")

In [ ]:
# =====================================================================
# --- 2. CONFIGURATION DES MODÈLES ---
# =====================================================================
# Hyperparamètres choisis pour un bon équilibre qualité / temps
# sur un graphe de taille moyenne (~600 entités, ~10 relations).
#
#   embedding_dim  : taille du vecteur de représentation (128 est standard)
#   epochs         : 200 est suffisant pour converger sur ce dataset
#   batch_size     : 256 pour stabiliser le gradient
#   lr             : 0.001 (Adam optimizer, valeur par défaut recommandée)
# =====================================================================

EMBEDDING_DIM = 128
EPOCHS        = 200
BATCH_SIZE    = 256
LR            = 0.001

MODEL_CONFIGS = {
    "TransE": {
        "model": "TransE",
        # TransE est le modèle fondateur (Bordes et al., 2013).
        # Il modélise chaque relation r comme une translation dans
        # l'espace vectoriel : h + r ≈ t.
        # Simple, efficace, mais peine sur les relations symétriques.
        "model_kwargs": {
            "embedding_dim": EMBEDDING_DIM,
        },
        "loss":    "MarginRankingLoss",
        "color":   "#E74C3C",
    },
    "ComplEx": {
        "model": "ComplEx",
        # ComplEx (Trouillon et al., 2016) utilise des embeddings dans
        # l'espace des nombres complexes. L'asymétrie est modélisée via
        # la partie imaginaire — parfait pour playsFor, coachedBy, etc.
        "model_kwargs": {
            "embedding_dim": EMBEDDING_DIM,
        },
        "loss":    "SoftplusLoss",
        "color":   "#2980B9",
    },
}

print("2. Configuration des modèles :")
for name, cfg in MODEL_CONFIGS.items():
    print(f"   {name:10s} | dim={EMBEDDING_DIM} | loss={cfg['loss']}")

In [ ]:
# =====================================================================
# --- 3. ENTRAÎNEMENT ---
# =====================================================================
results_store = {}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'─'*55}")
    print(f"  Entraînement : {model_name}")
    print(f"{'─'*55}")
    t_start = time.time()

    result = pipeline(
        # Données
        training=tf_train,
        validation=tf_valid,
        testing=tf_test,
        # Modèle
        model=cfg["model"],
        model_kwargs=cfg["model_kwargs"],
        # Optimisation
        optimizer="Adam",
        optimizer_kwargs={"lr": LR},
        loss=cfg["loss"],
        # Entraînement
        training_kwargs={
            "num_epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
        },
        # Évaluation (calculée automatiquement sur test après training)
        evaluation_kwargs={"batch_size": BATCH_SIZE},
        # Reproductibilité
        random_seed=42,
        device=device,
    )

    elapsed = time.time() - t_start
    results_store[model_name] = {
        "pipeline_result": result,
        "elapsed":         elapsed,
    }

    # Résumé rapide
    metrics = result.metric_results.to_flat_dict()
    mrr     = metrics.get("both.realistic.inverse_harmonic_mean_rank", 0)
    hits1   = metrics.get("both.realistic.hits_at_1", 0)
    hits10  = metrics.get("both.realistic.hits_at_10", 0)
    print(f"  ✅ {model_name} entraîné en {elapsed:.0f}s")
    print(f"     MRR      : {mrr:.4f}")
    print(f"     Hits@1   : {hits1:.4f}")
    print(f"     Hits@10  : {hits10:.4f}")

    # Sauvegarde du modèle
    save_dir = f"models/kge_{model_name.lower()}"
    os.makedirs(save_dir, exist_ok=True)
    result.save_to_directory(save_dir)
    print(f"     Sauvegardé : {save_dir}/")

In [ ]:
result_transe = results_store["TransE"]["pipeline_result"]
all_metrics = result_transe.metric_results.to_flat_dict()

print("=== Clés de métriques disponibles ===")
for key, val in sorted(all_metrics.items()):
    print(f"  {key:<60} : {val:.4f}")

In [ ]:
# =====================================================================
# --- 4. TABLEAU COMPARATIF DES MÉTRIQUES ---
# =====================================================================
print("\n4. Tableau comparatif des métriques (jeu de test)...\n")

METRIC_KEYS = {
    "MRR"     : "both.realistic.inverse_harmonic_mean_rank",
    "Hits@1"  : "both.realistic.hits_at_1",
    "Hits@3"  : "both.realistic.hits_at_3",
    "Hits@10" : "both.realistic.hits_at_10",
    "MR"      : "both.realistic.mean_rank",
}

rows = []
for model_name, data in results_store.items():
    metrics = data["pipeline_result"].metric_results.to_flat_dict()
    row = {"Modèle": model_name}
    for label, key in METRIC_KEYS.items():
        val = metrics.get(key, float("nan"))
        row[label] = round(float(val), 4)
    row["Temps (s)"] = round(data["elapsed"], 1)
    rows.append(row)

df_metrics = pd.DataFrame(rows).set_index("Modèle")
print(df_metrics.to_string())

# Sauvegarde CSV
df_metrics.to_csv("data/kge_metrics_comparison.csv")
print("\n   -> kge_metrics_comparison.csv sauvegardé")

In [ ]:
# =====================================================================
# --- 5. VISUALISATION : COURBES DE LOSS ---
# =====================================================================
print("\n5. Visualisation des courbes de loss...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("C.3 — KGE Training Loss : TransE vs ComplEx", fontsize=14, fontweight="bold")

for ax, (model_name, data) in zip(axes, results_store.items()):
    losses = data["pipeline_result"].losses
    color  = MODEL_CONFIGS[model_name]["color"]
    ax.plot(losses, color=color, linewidth=1.5, label=model_name)
    ax.set_title(model_name, fontsize=12)
    ax.set_xlabel("Époque")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    # Annotation de la loss finale
    ax.annotate(
        f"Final : {losses[-1]:.4f}",
        xy=(len(losses)-1, losses[-1]),
        xytext=(-60, 20), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="gray"),
        fontsize=9, color=color
    )

plt.tight_layout()
plt.savefig("graphs/kge_training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("   -> kge_training_loss.png sauvegardé")

In [ ]:
# =====================================================================
# --- 6. RÉSUMÉ C.3 ---
# =====================================================================
print(f"\n{'='*60}")
print("✅  C.3 TERMINÉ — Résumé de l'entraînement KGE")
print(f"{'='*60}")
print(f"  Entités    : {tf_train.num_entities:,}")
print(f"  Relations  : {tf_train.num_relations:,}")
print(f"  Train      : {tf_train.num_triples:,} triplets")
print(f"  Dim. embed : {EMBEDDING_DIM}")
print(f"  Epochs     : {EPOCHS}")
print()
for model_name in results_store:
    metrics = results_store[model_name]["pipeline_result"].metric_results.to_flat_dict()
    mrr  = metrics.get("both.realistic.inverse_harmonic_mean_rank", 0)
    h1   = metrics.get("both.realistic.hits_at_1", 0)
    h10  = metrics.get("both.realistic.hits_at_10", 0)
    t    = results_store[model_name]["elapsed"]
    print(f"  {model_name:10s} | MRR={mrr:.4f} | H@1={h1:.4f} | H@10={h10:.4f} | {t:.0f}s")

print(f"\n  Fichiers générés :")
print(f"   • kge_transe/       — modèle TransE sauvegardé")
print(f"   • kge_complex/      — modèle ComplEx sauvegardé")
print(f"   • kge_metrics_comparison.csv")
print(f"   • kge_training_loss.png")
print(f"\n  → Prochaine étape : C.4 — Évaluation approfondie & t-SNE")

Structure du C.3 :  
- Étape 1 — Chargement des splits avec TriplesFactory.from_path(). Le point clé est de partager entity_to_id et relation_to_id du train vers valid/test — c'est ce qui garantit l'absence d'entités OOV (déjà géré en C.1, mais PyKEEN l'enforces ici).
- Étape 2 — Configuration : deux modèles avec des approches fondamentalement différentes. TransE est translationnel (le plus simple, bon baseline), ComplEx est bilinéaire complexe (meilleur sur les relations asymétriques comme playsFor ou coachedBy).
- Étape 3 — pipeline() : c'est la fonction centrale de PyKEEN — elle orchestre entraînement + évaluation en une seule ligne. On passe directement les TriplesFactory, pas des chemins de fichiers.
- Étape 4 & 5 — Tableau comparatif et courbes de loss, qui seront utiles pour le rapport (section C.4 du grading).  


Quand tu as les résultats, on attaque C.4 — analyse approfondie des métriques + sensibilité à la taille + t-SNE.

Pourquoi TransE surpasse ComplEx ici ? Trois raisons probables à mentionner dans ton rapport :

- Ton graphe a 459 relations mais elles sont très déséquilibrées (playsFor domine massivement) — TransE gère mieux ce profil
- La loss de ComplEx (SoftplusLoss) est sur une échelle absolument différente (0→9 vs 0→0.9 pour TransE), ce qui suggère que les hyperparamètres (lr, batch_size) ne sont pas optimaux pour ComplEx — un tuning dédié l'améliorerait
- Le graphe est très sparse (38k entités, ~58k triplets = ~1.5 triplets/entité en moyenne) — TransE est plus robuste dans ce cas

## C.4 — Évaluation approfondie, Sensibilité à la taille & t-SNE

Cette section approfondit l'analyse des modèles entraînés en C.3 :
1. Tableau complet head vs tail (asymétrie du graphe)
2. Analyse de sensibilité : performance vs taille du training set
3. Visualisation t-SNE des embeddings d'entités
4. Plus proches voisins — validation qualitative


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
import warnings, time, os
warnings.filterwarnings("ignore")

print("Librairies chargées ✓")

In [ ]:
# =====================================================================
# --- 1. TABLEAU COMPLET HEAD / TAIL / BOTH ---
# =====================================================================
# PyKEEN évalue séparément la prédiction de la tête (?→r→t)
# et de la queue (h→r→?). Analyser les deux révèle si le modèle
# est meilleur pour retrouver des sujets ou des objets.
# =====================================================================
print("1. Tableau complet des métriques (head / tail / both)...\n")

DIRECTIONS = ["head", "tail", "both"]
METRICS_LABELS = {
    "MRR"     : "inverse_harmonic_mean_rank",
    "Hits@1"  : "hits_at_1",
    "Hits@3"  : "hits_at_3",
    "Hits@10" : "hits_at_10",
    "MR"      : "arithmetic_mean_rank",
}

rows_full = []
for model_name, data in results_store.items():
    flat = data["pipeline_result"].metric_results.to_flat_dict()
    for direction in DIRECTIONS:
        row = {"Modèle": model_name, "Direction": direction}
        for label, key_suffix in METRICS_LABELS.items():
            full_key = f"{direction}.realistic.{key_suffix}"
            val = flat.get(full_key, float("nan"))
            row[label] = round(float(val), 4)
        rows_full.append(row)

df_full = pd.DataFrame(rows_full).set_index(["Modèle", "Direction"])
print(df_full.to_string())
df_full.to_csv("data/kge_metrics_full.csv")
print("\n   -> kge_metrics_full.csv sauvegardé")

In [ ]:
# =====================================================================
# --- 2. ANALYSE DE SENSIBILITÉ À LA TAILLE ---
# =====================================================================
# On ré-entraîne chaque modèle sur des sous-ensembles du training set
# (20k et 50k triplets) puis on compare les métriques avec le full.
# Cela montre comment la qualité des embeddings évolue avec la taille
# du KG — information clé pour le rapport.
#
# Note : on utilise les mêmes hyperparamètres qu'en C.3 pour une
# comparaison équitable. Seule la taille du training set varie.
# =====================================================================
print("\n2. Analyse de sensibilité à la taille du graphe...")
print("   (ré-entraînement sur 20k et 50k — patientez ~10 min)\n")

SIZES = {
    "20k" : 20_000,
    "50k" : 50_000,
    "full": tf_train.num_triples,  # ~58k
}

sensitivity_results = {}  # {model: {size_label: metrics_dict}}

for model_name, cfg in MODEL_CONFIGS.items():
    sensitivity_results[model_name] = {}

    # Résultats "full" déjà disponibles depuis C.3
    flat_full = results_store[model_name]["pipeline_result"].metric_results.to_flat_dict()
    sensitivity_results[model_name]["full"] = flat_full

    for size_label, n_triples in [("20k", 20_000), ("50k", 50_000)]:
        print(f"   [{model_name}] Taille : {size_label} ({n_triples:,} triplets)...")

        # Sous-échantillonnage aléatoire reproductible
        np.random.seed(42)
        idx = np.random.choice(tf_train.num_triples, size=n_triples, replace=False)
        mapped = tf_train.mapped_triples[idx]

        tf_sub = TriplesFactory(
            mapped_triples=mapped,
            entity_to_id=tf_train.entity_to_id,
            relation_to_id=tf_train.relation_to_id,
        )

        t0 = time.time()
        result_sub = pipeline(
            training=tf_sub,
            validation=tf_valid,
            testing=tf_test,
            model=cfg["model"],
            model_kwargs=cfg["model_kwargs"],
            optimizer="Adam",
            optimizer_kwargs={"lr": LR},
            loss=cfg["loss"],
            training_kwargs={"num_epochs": EPOCHS, "batch_size": BATCH_SIZE},
            evaluation_kwargs={"batch_size": BATCH_SIZE},
            random_seed=42,
            device=device,
        )
        elapsed = time.time() - t0

        flat = result_sub.metric_results.to_flat_dict()
        sensitivity_results[model_name][size_label] = flat

        mrr  = flat.get("both.realistic.inverse_harmonic_mean_rank", 0)
        h10  = flat.get("both.realistic.hits_at_10", 0)
        print(f"      ✓ MRR={mrr:.4f} | H@10={h10:.4f} | {elapsed:.0f}s")

print("\n   ✅ Analyse de sensibilité terminée.")

In [ ]:
# ── À ajouter à la fin de l'étape 2 (sensibilité) ───────────────────
import json

# Sauvegarder les métriques de sensibilité en JSON pour persistance
sensitivity_to_save = {}
for model_name, size_dict in sensitivity_results.items():
    sensitivity_to_save[model_name] = {}
    for size_label, flat in size_dict.items():
        sensitivity_to_save[model_name][size_label] = {
            k: float(v) for k, v in flat.items()
        }

with open("data/sensitivity_results.json", "w") as f:
    json.dump(sensitivity_to_save, f, indent=2)
print("   -> sensitivity_results.json sauvegardé sur Drive ✓")

TransE montre une progression quasi-linéaire avec la taille du graphe, ce qui confirme sa robustesse sur les données sparse. ComplEx reste très sensible à la quantité de données et nécessiterait un tuning d'hyperparamètres dédié (learning rate plus faible, regularisation) pour rivaliser avec TransE sur ce graphe.

In [ ]:
# =====================================================================
# --- 3. VISUALISATION DE LA SENSIBILITÉ ---
# =====================================================================
print("\n3. Visualisation de la sensibilité à la taille...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("C.4 — Sensibilité des KGE à la taille du graphe", fontsize=14, fontweight="bold")

SIZE_ORDER  = ["20k", "50k", "full"]
SIZE_VALUES = [20_000, 50_000, tf_train.num_triples]

for ax, (model_name, cfg) in zip(axes, MODEL_CONFIGS.items()):
    color = cfg["color"]
    size_results = sensitivity_results[model_name]

    mrr_vals  = [size_results[s].get("both.realistic.inverse_harmonic_mean_rank", 0) for s in SIZE_ORDER]
    h1_vals   = [size_results[s].get("both.realistic.hits_at_1", 0)   for s in SIZE_ORDER]
    h10_vals  = [size_results[s].get("both.realistic.hits_at_10", 0)  for s in SIZE_ORDER]

    x = SIZE_VALUES
    ax.plot(x, mrr_vals,  "o-",  color=color,   linewidth=2, label="MRR",     markersize=8)
    ax.plot(x, h1_vals,   "s--", color=color,   linewidth=1.5, label="Hits@1", markersize=7, alpha=0.7)
    ax.plot(x, h10_vals,  "^-.", color=color,   linewidth=1.5, label="Hits@10",markersize=7, alpha=0.7)

    ax.set_title(model_name, fontsize=12)
    ax.set_xlabel("Taille du training set (triplets)")
    ax.set_ylabel("Score")
    ax.set_xticks(SIZE_VALUES)
    ax.set_xticklabels(SIZE_ORDER)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(max(h10_vals) * 1.2, 0.05))

plt.tight_layout()
plt.savefig("graphs/kge_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("   -> kge_sensitivity.png sauvegardé")

In [ ]:
# =====================================================================
# --- 4. VISUALISATION t-SNE DES EMBEDDINGS ---
# =====================================================================
# t-SNE projette les embeddings de dim 128 → dim 2 pour révéler
# la structure géométrique apprise par le modèle.
# On colorie par type d'entité (Person, Team, Country…) pour voir
# si le modèle a appris à regrouper les entités similaires.
# =====================================================================
print("\n4. Visualisation t-SNE (sur TransE — meilleur modèle)...")

# Extraction des embeddings TransE
transe_model = results_store["TransE"]["pipeline_result"].model
entity_embeddings = transe_model.entity_representations[0](
    indices=None
).detach().cpu().numpy()

print(f"   -> Embeddings extraits : {entity_embeddings.shape}  (entités × dim)")

# Récupération du mapping id → nom d'entité
id_to_entity = {v: k for k, v in tf_train.entity_to_id.items()}

# Classification des entités par type (depuis le nom de l'URI)
EX_BASE = "http://www.example.org/football/"

def get_entity_type(uri):
    """Déduit le type à partir du graphe augmenté ou du nom de l'URI."""
    name = uri.replace(EX_BASE, "")
    # Heuristiques simples basées sur la structure du KG
    if any(suffix in name for suffix in ["_FC", "_AFC", "_United", "_City",
                                          "_Hotspur", "_Athletic", "_Wanderers",
                                          "Bournemouth", "Brighton", "Brentford",
                                          "Chelsea", "Arsenal", "Liverpool",
                                          "Palace", "Everton", "Fulham",
                                          "Nottingham", "Sunderland", "Ipswich",
                                          "Leicester", "Southampton", "Leeds"]):
        return "Team"
    if any(c.islower() for c in name) and "_" in name and len(name) > 6:
        return "Person"
    if name[0].isupper() and "_" not in name:
        return "Country/Position"
    return "Other"

# Application du typage
n_entities = entity_embeddings.shape[0]
entity_types = [get_entity_type(id_to_entity.get(i, "")) for i in range(n_entities)]

type_to_color = {
    "Person"           : "#3498DB",
    "Team"             : "#E74C3C",
    "Country/Position" : "#2ECC71",
    "Other"            : "#95A5A6",
}

# t-SNE (sur un échantillon de 3000 entités max pour la lisibilité)
N_SAMPLE = min(3000, n_entities)
np.random.seed(42)
sample_idx = np.random.choice(n_entities, size=N_SAMPLE, replace=False)

emb_sample   = entity_embeddings[sample_idx]
types_sample = [entity_types[i] for i in sample_idx]

print(f"   -> t-SNE sur {N_SAMPLE} entités (dim 128 → 2)... (patientez ~1 min)")
tsne = TSNE(
    n_components=2,
    perplexity=40,
    n_iter=1000,
    random_state=42,
    init="pca",
    learning_rate="auto",
)
emb_2d = tsne.fit_transform(emb_sample)
print("   -> t-SNE terminé ✓")

# Plot
fig, ax = plt.subplots(figsize=(13, 9))
fig.suptitle("C.4 — t-SNE des embeddings TransE (dim 128 → 2D)", fontsize=14, fontweight="bold")

for etype, color in type_to_color.items():
    mask = [i for i, t in enumerate(types_sample) if t == etype]
    if mask:
        ax.scatter(
            emb_2d[mask, 0], emb_2d[mask, 1],
            c=color, label=etype,
            s=12, alpha=0.6, linewidths=0
        )

# Annotation des équipes (identifiables facilement)
team_indices = [i for i, t in enumerate(types_sample) if t == "Team"]
for idx in team_indices[:15]:  # annoter les 15 premières équipes
    entity_uri = id_to_entity.get(sample_idx[idx], "")
    name = entity_uri.replace(EX_BASE, "").replace("_FC", "").replace("_", " ")
    ax.annotate(name, (emb_2d[idx, 0], emb_2d[idx, 1]),
                fontsize=7, alpha=0.85,
                xytext=(4, 4), textcoords="offset points")

ax.legend(fontsize=11, markerscale=2)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("graphs/kge_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("   -> kge_tsne.png sauvegardé")

In [ ]:
# =====================================================================
# --- 5. PLUS PROCHES VOISINS (VALIDATION QUALITATIVE) ---
# =====================================================================
# On vérifie que le modèle a appris des représentations sensées :
# les voisins d'une équipe doivent être d'autres équipes,
# les voisins d'un joueur doivent être des joueurs proches.
# =====================================================================
print("\n5. Plus proches voisins — validation qualitative...")

def nearest_neighbors(entity_uri, embeddings, entity_to_id, id_to_entity, top_k=5):
    """Retourne les top_k entités les plus proches dans l'espace des embeddings."""
    if entity_uri not in entity_to_id:
        return []
    idx    = entity_to_id[entity_uri]
    vec    = embeddings[idx].reshape(1, -1)
    sims   = cosine_similarity(vec, embeddings)[0]
    sims[idx] = -1  # exclure l'entité elle-même
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(id_to_entity[i], round(float(sims[i]), 4)) for i in top_idx]

# Entités de référence à tester
PROBE_ENTITIES = [
    f"{EX_BASE}Arsenal_FC",
    f"{EX_BASE}Chelsea_FC",
    f"{EX_BASE}England",
    f"{EX_BASE}Mohamed_Salah",
]

print()
for probe_uri in PROBE_ENTITIES:
    name = probe_uri.replace(EX_BASE, "").replace("_", " ")
    neighbors = nearest_neighbors(
        probe_uri, entity_embeddings,
        tf_train.entity_to_id, id_to_entity, top_k=5
    )
    if neighbors:
        print(f"  🔍 Voisins de '{name}' :")
        for neighbor_uri, sim in neighbors:
            neighbor_name = neighbor_uri.replace(EX_BASE, "").replace("_", " ")
            print(f"      {neighbor_name:<35} (sim={sim:.4f})")
    else:
        print(f"  ⚠️  '{name}' non trouvée dans le graphe.")
    print()

In [ ]:
# =====================================================================
# --- 6. RÉSUMÉ C.4 ---
# =====================================================================
print(f"\n{'='*60}")
print("✅  C.4 TERMINÉ — Résumé de l'évaluation KGE")
print(f"{'='*60}")

print("\n  Métriques finales (jeu de test complet) :")
print(f"  {'Modèle':<12} {'MRR':>8} {'Hits@1':>8} {'Hits@3':>8} {'Hits@10':>8}")
print(f"  {'─'*44}")
for model_name in results_store:
    flat = results_store[model_name]["pipeline_result"].metric_results.to_flat_dict()
    mrr = flat.get("both.realistic.inverse_harmonic_mean_rank", 0)
    h1  = flat.get("both.realistic.hits_at_1", 0)
    h3  = flat.get("both.realistic.hits_at_3", 0)
    h10 = flat.get("both.realistic.hits_at_10", 0)
    print(f"  {model_name:<12} {mrr:>8.4f} {h1:>8.4f} {h3:>8.4f} {h10:>8.4f}")

print(f"\n  Sensibilité à la taille :")
print(f"  {'Modèle':<12} {'20k MRR':>10} {'50k MRR':>10} {'full MRR':>10}")
print(f"  {'─'*44}")
for model_name in sensitivity_results:
    vals = [sensitivity_results[model_name][s].get(
        "both.realistic.inverse_harmonic_mean_rank", 0)
        for s in ["20k", "50k", "full"]]
    print(f"  {model_name:<12} {vals[0]:>10.4f} {vals[1]:>10.4f} {vals[2]:>10.4f}")

print(f"\n  Fichiers générés :")
print(f"   • kge_metrics_full.csv    — métriques head/tail/both")
print(f"   • kge_sensitivity.png     — courbes de sensibilité")
print(f"   • kge_tsne.png            — projection t-SNE")
print(f"\n  → C.3 et C.4 sont complets. Partie C terminée !")
print(f"  → Prochaine étape : D — RAG over RDF/SPARQL")

Ce que produit ce C.4 :
- Étape 1 — Tableau head/tail/both séparé : révèle si le modèle prédit mieux les têtes ou les queues (dans ton graphe, la prédiction de queue sera meilleure car playsFor a des tails répétitifs comme les équipes).
- Étape 2 & 3 — Sensibilité à la taille : re-entraîne sur 20k et 50k. C'est le point qui prend le plus de temps (~10 min supplémentaires avec GPU). Si tu es pressé, tu peux réduire les epochs à 100 pour ces sous-expériences.
- Étape 4 — t-SNE : la visualisation clé du rapport. Tu devrais voir les équipes former un cluster distinct des joueurs.
- Étape 5 — Plus proches voisins : validation qualitative — si Arsenal et Chelsea sont voisins, le modèle a bien appris la structure "équipe de Premier League".

Analyse des résultats C.4
Graphe de sensibilité — TransE montre une progression régulière et prévisible, ce qui en fait un modèle fiable même avec peu de données. ComplEx explose seulement au "full", ce qui suggère qu'il a besoin d'un volume critique de triplets pour commencer à apprendre correctement.
t-SNE & plus proches voisins — les résultats sont sémantiquement très solides :

Arsenal → Leeds, Man City, Brighton… ✅ que des clubs de Premier League — le modèle a appris la notion d'équipe
England → Sweden, Wales, Norway, Serbia… ✅ que des pays européens — la notion géographique/nationale est bien capturée
Mohamed Salah → Ismaïla Sarr, Lorenzo Lucca… ✅ des attaquants africains/internationaux — cohérent avec le profil joueur

Deux petits artefacts à mentionner dans le rapport : un joueur (Josh Acheampong) apparaît dans les voisins de Chelsea car il joue pour eux, et une URI Wikidata brute apparaît — ce sont des traces de l'alignement ontologique fait en partie B.

=========================================================

Bilan de toute la partie C
- C.1 : 58k train / 3.2k valid / 3.2k test, 38k entités, 459 relations
- C.2 : 2 règles SWRL → 647 competesIn + 647 coachedBy inférés
- C.3 : TransE (MRR=0.16) >> ComplEx (MRR=0.02), convergence confirmée
- C.4 : Sensibilité croissante, clusters sémantiques cohérents en t-SNE

# CELLULE DE SECOURS — À exécuter UNIQUEMENT après un reset

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE DE SECOURS — À exécuter UNIQUEMENT après un reset
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, torch, numpy as np, pandas as pd, json, warnings
warnings.filterwarnings("ignore")

WORK_DIR = "/content/drive/MyDrive/Project Web Datamining & Semantics/Part C"
os.chdir(WORK_DIR)

# ── Librairies KGE ───────────────────────────────────────
from pykeen.triples import TriplesFactory
from pykeen.pipeline import PipelineResult

# ── Rechargement des splits ──────────────────────────────
tf_train = TriplesFactory.from_path("train.txt", delimiter="\t")
tf_valid = TriplesFactory.from_path("valid.txt", delimiter="\t",
               entity_to_id=tf_train.entity_to_id,
               relation_to_id=tf_train.relation_to_id)
tf_test  = TriplesFactory.from_path("test.txt",  delimiter="\t",
               entity_to_id=tf_train.entity_to_id,
               relation_to_id=tf_train.relation_to_id)
print(f"Splits rechargés → train={tf_train.num_triples:,} | "
      f"valid={tf_valid.num_triples:,} | test={tf_test.num_triples:,}")

# ── Rechargement des modèles (sans réentraîner) ──────────
result_transe  = PipelineResult.from_directory("models/kge_transe")
result_complex = PipelineResult.from_directory("models/kge_complex")
results_store  = {
    "TransE" : {"pipeline_result": result_transe,  "elapsed": 318.0},
    "ComplEx": {"pipeline_result": result_complex, "elapsed": 604.3},
}
print("Modèles rechargés → TransE ✓ | ComplEx ✓")

# ── Rechargement des résultats de sensibilité ────────────
# (seulement si l'étape 2 de C.4 a déjà été exécutée)
if os.path.exists("sensitivity_results.json"):
    with open("sensitivity_results.json") as f:
        sensitivity_results = json.load(f)
    print("sensitivity_results rechargé ✓")
else:
    print("⚠️  sensitivity_results.json absent — relancer l'étape 2 de C.4")

# ── Hyperparamètres (doivent correspondre à C.3) ─────────
EMBEDDING_DIM = 128
EPOCHS        = 200
BATCH_SIZE    = 256
LR            = 0.001
device        = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_CONFIGS = {
    "TransE" : {"model": "TransE",  "model_kwargs": {"embedding_dim": EMBEDDING_DIM},
                "loss": "MarginRankingLoss", "color": "#E74C3C"},
    "ComplEx": {"model": "ComplEx", "model_kwargs": {"embedding_dim": EMBEDDING_DIM},
                "loss": "SoftplusLoss",      "color": "#2980B9"},
}

print(f"\n✅ Tout rechargé depuis Drive. Device : {device}")
print(f"   Fichiers disponibles : {os.listdir(WORK_DIR)}")